# SET OS · Camera Coach Stage-2 silver-action fit

This notebook is orchestration only. It mounts Google Drive, verifies one deterministic code bundle and one derivative-only Stage-2 data bundle, checks the completed Stage-1 encoder binding, and runs `train_silver_actions.py` on the three extracted paired-corruption roots. Put the large data ZIP directly in the documented Google Drive `WORK_ROOT` before running the data verification cell; it intentionally does not use browser `files.upload()` for that payload. The run is research/silver only: `human_gold=false`, `release_admissible=false`, fit metrics only, and no source/control image is used as evaluation data.

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile

# Paste the two external bundle hashes and the completed Stage-1 artifact hash before running.
EXPECTED_BUNDLE_SHA256 = ''
EXPECTED_DATA_BUNDLE_SHA256 = ''
EXPECTED_STAGE1_ENCODER_SHA256 = ''
RESUME = False
MAX_MEMBER_BYTES = 2 * 1024 * 1024
MAX_TOTAL_BYTES = 8 * 1024 * 1024
MAX_DATA_MEMBER_BYTES = 16 * 1024 * 1024
MAX_DATA_TOTAL_BYTES = 2 * 1024 * 1024 * 1024
MAX_DATA_MEMBERS = 20_000
EXPECTED_BUNDLE_FILES = [
    'datasets/camera-coach/v1/silver-action-pair-schema.json',
    'datasets/camera-coach/v1/silver-geometry-schema.json',
    'ml/camera_coach/configs/silver_actions_stage2_colab.json',
    'ml/camera_coach/contracts/set_composition_net_v1.json',
    'ml/camera_coach/colab/SET_OS_Camera_Coach_Stage2.ipynb',
    'ml/camera_coach/data/__init__.py',
    'ml/camera_coach/data/preprocessing.py',
    'ml/camera_coach/losses.py',
    'ml/camera_coach/models/__init__.py',
    'ml/camera_coach/models/set_composition_net.py',
    'tools/dataset/extract_camera_geometry.swift',
    'tools/dataset/generate_camera_corruptions.py',
    'ml/camera_coach/train_silver_actions.py',
]
BUNDLE_MANIFEST_SCHEMA = 'camera-coach-stage2-colab-bundle-v1'
BUNDLE_SUMS_SCHEMA = 'camera-coach-stage2-sha256-v1'
DATA_BUNDLE_MANIFEST_SCHEMA = 'camera-coach-stage2-data-bundle-v1'
DATA_BUNDLE_SUMS_SCHEMA = 'camera-coach-stage2-data-sha256-v1'
EXPECTED_DATA_ROOTS = [
    {'id': 'commons', 'archive_root': 'pairs/commons', 'receipt_sha256': 'bab422ec74c47852866ffa37c60ae27121ede045a239683e2969a61d9e2c874a', 'pairs': 233},
    {'id': 'eva', 'archive_root': 'pairs/eva', 'receipt_sha256': 'ec40169d8d58a96c80652365c04e8eeae88ad5fa07e479c445cab47e92d34680', 'pairs': 1099},
    {'id': 'aadb', 'archive_root': 'pairs/aadb', 'receipt_sha256': '85ae7d6849895a721d1aceba9df0dd3db0046cc302d4edc6e0470919d7c312ce', 'pairs': 4265},
]

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/SET_OS/Camera_Coach_STAGE2')
WORK_ROOT = DRIVE_ROOT / 'workspace'
BUNDLE_PATH = WORK_ROOT / 'SET_OS_Camera_Coach_Stage2.zip'
DATA_BUNDLE_PATH = WORK_ROOT / 'SET_OS_CAMERA_STAGE2_DATA_20260909_v1.zip'
EXTRACT_ROOT = WORK_ROOT / 'bundle'
DATA_EXTRACT_ROOT = WORK_ROOT / 'pair-data'
# Pair roots are published from the verified single data archive.
PAIR_ROOT_SPECS = [
    {'root': DATA_EXTRACT_ROOT / 'pairs/commons', 'receipt_sha256': 'bab422ec74c47852866ffa37c60ae27121ede045a239683e2969a61d9e2c874a'},
    {'root': DATA_EXTRACT_ROOT / 'pairs/eva', 'receipt_sha256': 'ec40169d8d58a96c80652365c04e8eeae88ad5fa07e479c445cab47e92d34680'},
    {'root': DATA_EXTRACT_ROOT / 'pairs/aadb', 'receipt_sha256': '85ae7d6849895a721d1aceba9df0dd3db0046cc302d4edc6e0470919d7c312ce'},
]
PAIR_ROOTS = [spec['root'] for spec in PAIR_ROOT_SPECS]
PAIR_RECEIPT_SHA256S = [spec['receipt_sha256'] for spec in PAIR_ROOT_SPECS]
STAGE1_ROOT = Path('/content/drive/MyDrive/SET_OS/EVA_STAGE1/run')
RUN_ROOT = DRIVE_ROOT / 'stage2-run'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('persistent pair roots:', PAIR_ROOTS)
print('Stage-1 root:', STAGE1_ROOT)
print('restartable Stage-2 run root:', RUN_ROOT)

In [ ]:
# Upload the code bundle produced by package_camera_colab.py --profile stage2.
if not BUNDLE_PATH.is_file():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(f'expected exactly one bundle upload, got {len(uploaded)}')
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    if not uploaded_name.lower().endswith('.zip'):
        raise RuntimeError('uploaded file must be a .zip bundle')
    BUNDLE_PATH.write_bytes(uploaded_bytes)
print(BUNDLE_PATH, BUNDLE_PATH.stat().st_size, 'bytes')
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_BUNDLE_SHA256):
    raise RuntimeError('paste the lowercase bundle SHA-256 into EXPECTED_BUNDLE_SHA256')
actual_bundle_sha256 = sha256_file(BUNDLE_PATH)
if actual_bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(f'whole bundle SHA-256 mismatch: expected {EXPECTED_BUNDLE_SHA256}, got {actual_bundle_sha256}')

In [ ]:
# Place SET_OS_CAMERA_STAGE2_DATA_20260909_v1.zip directly in Google Drive WORK_ROOT before running this cell.
# The large data payload deliberately bypasses browser upload to keep it out of Colab RAM.
if not DATA_BUNDLE_PATH.is_file():
    raise FileNotFoundError(f'Place the data ZIP at {DATA_BUNDLE_PATH} in Google Drive, then rerun this cell')
if DATA_BUNDLE_PATH.is_symlink() or not DATA_BUNDLE_PATH.is_file():
    raise RuntimeError('Stage-2 data path must be a regular file, not a symlink')
print(DATA_BUNDLE_PATH, DATA_BUNDLE_PATH.stat().st_size, 'bytes')
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_DATA_BUNDLE_SHA256):
    raise RuntimeError('paste the lowercase data bundle SHA-256 into EXPECTED_DATA_BUNDLE_SHA256')
actual_data_bundle_sha256 = sha256_file(DATA_BUNDLE_PATH)
if actual_data_bundle_sha256 != EXPECTED_DATA_BUNDLE_SHA256:
    raise RuntimeError(f'whole data bundle SHA-256 mismatch: expected {EXPECTED_DATA_BUNDLE_SHA256}, got {actual_data_bundle_sha256}')

In [ ]:
# Verify and atomically publish the derivative-only data archive.
if DATA_BUNDLE_PATH.stat().st_size > MAX_DATA_TOTAL_BYTES:
    raise RuntimeError('data bundle exceeds total size ceiling')
data_staging = Path(tempfile.mkdtemp(prefix='.pair-data-staging-', dir=WORK_ROOT))
data_staging_ready = True
try:
    with zipfile.ZipFile(DATA_BUNDLE_PATH) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise RuntimeError('data bundle contains duplicate member names')
        for metadata_name in ('bundle-manifest.json', 'SHA256SUMS.json'):
            metadata_info = archive.getinfo(metadata_name)
            if metadata_info.file_size > MAX_DATA_MEMBER_BYTES or metadata_info.compress_type != zipfile.ZIP_STORED:
                raise RuntimeError(f'unsafe data bundle metadata member: {metadata_name}')
        data_manifest = json.loads(archive.read('bundle-manifest.json'))
        data_sums = json.loads(archive.read('SHA256SUMS.json'))
        if data_manifest.get('schema_id') != DATA_BUNDLE_MANIFEST_SCHEMA or data_manifest.get('schema_version') != '1.0.0' or data_manifest.get('kind') != 'research_only_paired_corruption_data':
            raise RuntimeError('unsupported Stage-2 data bundle manifest')
        if data_manifest.get('research_only') is not True or data_manifest.get('human_gold') is not False or data_manifest.get('release_admissible') is not False or data_manifest.get('split') != 'research_fit':
            raise RuntimeError('data bundle crossed the research-only boundary')
        data_roots = data_manifest.get('roots')
        if not isinstance(data_roots, list) or [{key: root.get(key) for key in ('id', 'archive_root', 'receipt_sha256', 'pairs')} for root in data_roots] != EXPECTED_DATA_ROOTS:
            raise RuntimeError('data bundle root allowlist/count/hash mismatch')
        data_records = data_manifest.get('files')
        if not isinstance(data_records, list) or data_sums.get('schema_id') != DATA_BUNDLE_SUMS_SCHEMA or data_sums.get('files') != data_records:
            raise RuntimeError('data bundle SHA-256 manifest mismatch')
        if any(not isinstance(record, dict) or set(record) != {'path', 'bytes', 'sha256'} or type(record['bytes']) is not int or record['bytes'] < 0 or not re.fullmatch(r'[0-9a-f]{64}', record['sha256']) for record in data_records):
            raise RuntimeError('data bundle file records are malformed')
        expected_names = [record['path'] for record in data_records] + ['bundle-manifest.json', 'SHA256SUMS.json']
        if names != expected_names or len(names) > MAX_DATA_MEMBERS:
            raise RuntimeError('data bundle member order/allowlist mismatch')
        data_totals = data_manifest.get('totals')
        if not isinstance(data_totals, dict) or data_totals.get('members') != len(names) or data_totals.get('uncompressed_bytes') != sum(record['bytes'] for record in data_records):
            raise RuntimeError('data bundle totals mismatch')
        if sum(info.file_size for info in archive.infolist()) > MAX_DATA_TOTAL_BYTES:
            raise RuntimeError('data bundle members exceed total size ceiling')
        record_by_name = {record['path']: record for record in data_records}
        if len(record_by_name) != len(data_records):
            raise RuntimeError('data bundle file records contain duplicates')
        for info in archive.infolist():
            name = info.filename
            if name in {'bundle-manifest.json', 'SHA256SUMS.json'}:
                record = None
            else:
                record = record_by_name.get(name)
            relative = PurePosixPath(name)
            if name != relative.as_posix() or not name or '\\' in name or '\x00' in name or relative.is_absolute() or any(part in {'', '.', '..'} for part in relative.parts) or ':' in relative.parts[0] or info.is_dir():
                raise RuntimeError(f'unsafe data bundle member path: {name}')
            mode = (info.external_attr >> 16) & 0o170000
            if mode != stat.S_IFREG or info.compress_type != zipfile.ZIP_STORED:
                raise RuntimeError(f'non-regular or compressed data bundle member: {name}')
            if info.file_size > MAX_DATA_MEMBER_BYTES or (record is not None and info.file_size != record['bytes']):
                raise RuntimeError(f'unsafe or oversized data bundle member: {name}')
            target = data_staging.joinpath(*relative.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            digest = hashlib.sha256()
            count = 0
            with archive.open(info, 'r') as source, target.open('wb') as destination:
                for chunk in iter(lambda: source.read(1024 * 1024), b''):
                    destination.write(chunk)
                    digest.update(chunk)
                    count += len(chunk)
            if record is not None and (count != record['bytes'] or digest.hexdigest() != record['sha256']):
                raise RuntimeError(f'data bundle member hash mismatch: {name}')
    old_data_root = None
    if DATA_EXTRACT_ROOT.exists() or DATA_EXTRACT_ROOT.is_symlink():
        if DATA_EXTRACT_ROOT.is_symlink() or not DATA_EXTRACT_ROOT.is_dir():
            raise RuntimeError('existing pair-data root is not a regular directory')
        old_data_root = WORK_ROOT / f'.pair-data-old-{os.getpid()}'
        if old_data_root.exists() or old_data_root.is_symlink():
            raise RuntimeError('pair-data replacement collision')
        os.replace(DATA_EXTRACT_ROOT, old_data_root)
    try:
        os.replace(data_staging, DATA_EXTRACT_ROOT)
        data_staging_ready = False
    except Exception:
        if old_data_root is not None and not DATA_EXTRACT_ROOT.exists():
            os.replace(old_data_root, DATA_EXTRACT_ROOT)
        raise
    if old_data_root is not None:
        shutil.rmtree(old_data_root)
    print('verified and published data bundle', actual_data_bundle_sha256, 'roots:', [root['id'] for root in data_manifest['roots']], 'files:', len(data_records))
finally:
    if data_staging_ready and data_staging.exists():
        shutil.rmtree(data_staging)

In [ ]:
# Verify embedded allowlists and hashes before extraction.
if BUNDLE_PATH.stat().st_size > MAX_TOTAL_BYTES:
    raise RuntimeError('bundle exceeds total size ceiling')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    manifest = json.loads(archive.read('bundle-manifest.json'))
    sums = json.loads(archive.read('SHA256SUMS.json'))
    if manifest.get('schema_id') != BUNDLE_MANIFEST_SCHEMA or manifest.get('schema_version') != '1.0.0' or manifest.get('kind') != 'research_only_colab_orchestration':
        raise RuntimeError('unsupported Stage-2 bundle manifest')
    records = manifest.get('files')
    if [record.get('path') for record in records] != EXPECTED_BUNDLE_FILES:
        raise RuntimeError('Stage-2 bundle source allowlist mismatch')
    if sums.get('schema_id') != BUNDLE_SUMS_SCHEMA or sums.get('files') != records:
        raise RuntimeError('Stage-2 bundle SHA-256 manifest mismatch')
    expected_names = [*EXPECTED_BUNDLE_FILES, 'bundle-manifest.json', 'SHA256SUMS.json']
    if archive.namelist() != expected_names:
        raise RuntimeError('bundle member order/allowlist mismatch')
    total_uncompressed = sum(info.file_size for info in archive.infolist())
    if total_uncompressed > MAX_TOTAL_BYTES:
        raise RuntimeError('bundle members exceed total size ceiling')
    for info in archive.infolist():
        relative = Path(info.filename)
        if info.file_size > MAX_MEMBER_BYTES or relative.is_absolute() or '..' in relative.parts or info.filename.startswith('/'):
            raise RuntimeError(f'unsafe or oversized bundle member: {info.filename}')
        mode = (info.external_attr >> 16) & 0o170000
        if mode != stat.S_IFREG:
            raise RuntimeError(f'non-regular bundle member: {info.filename}')
        target = EXTRACT_ROOT.joinpath(*relative.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(info.filename))
    for record in records:
        payload = (EXTRACT_ROOT / record['path']).read_bytes()
        if len(payload) != record['bytes'] or hashlib.sha256(payload).hexdigest() != record['sha256']:
            raise RuntimeError(f"bundle member hash mismatch: {record['path']}")
print('verified bundle', actual_bundle_sha256, 'and', len(records), 'allowlisted source files')

In [ ]:
# Torch is supplied by the Colab runtime; Pillow is the existing image dependency.
# Install only the existing dependency when the runtime needs the pinned decoder.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', 'Pillow==12.2.0'], check=True)
import torch
from PIL import Image
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'pillow': getattr(Image, '__version__', 'unknown'), 'bundle_sha256': actual_bundle_sha256})

In [ ]:
# Every pair root is a separate trust domain. Require a complete root/hash list and verify each receipt before training.
if not PAIR_ROOT_SPECS or len(PAIR_ROOT_SPECS) != len(PAIR_ROOTS) or len(PAIR_ROOTS) != len(PAIR_RECEIPT_SHA256S):
    raise RuntimeError('PAIR_ROOT_SPECS must contain a complete (Drive root, receipt SHA-256) entry for every root')
for index, (pair_root, receipt_sha256) in enumerate(zip(PAIR_ROOTS, PAIR_RECEIPT_SHA256S)):
    if not isinstance(pair_root, Path) or not isinstance(receipt_sha256, str) or not re.fullmatch(r'[0-9a-f]{64}', receipt_sha256):
        raise RuntimeError(f'pair root {index} is missing its lowercase expected receipt SHA-256')
    pair_receipt_path = pair_root / 'receipt.json'
    if not pair_root.is_dir() or not (pair_root / 'pairs.jsonl').is_file() or not pair_receipt_path.is_file() or not (pair_root / 'images').is_dir():
        raise RuntimeError(f'pair root {index} is partial; require pairs.jsonl, receipt.json, and images/')
    actual_receipt_sha256 = hashlib.sha256(pair_receipt_path.read_bytes()).hexdigest()
    if actual_receipt_sha256 != receipt_sha256:
        raise RuntimeError(f'pair root {index} receipt SHA-256 mismatch: expected {receipt_sha256}, got {actual_receipt_sha256}')
# The Stage-1 receipt is the next trust anchor; the trainer checks it again before loading.
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_STAGE1_ENCODER_SHA256):
    raise RuntimeError('paste the completed Stage-1 final_artifact.sha256 into EXPECTED_STAGE1_ENCODER_SHA256')
stage1_receipt_path = STAGE1_ROOT / 'receipt.json'
stage1_receipt = json.loads(stage1_receipt_path.read_text(encoding='utf-8'))
if stage1_receipt.get('status') != 'complete' or stage1_receipt.get('final_artifact', {}).get('path') != 'encoder-final.pt' or stage1_receipt.get('final_artifact', {}).get('sha256') != EXPECTED_STAGE1_ENCODER_SHA256:
    raise RuntimeError('Stage-1 receipt does not bind the declared encoder hash')
if stage1_receipt.get('human_gold') is not False or stage1_receipt.get('release_admissible') is not False:
    raise RuntimeError('Stage-1 receipt crossed the research-only boundary')
print('Stage-1 and all silver-pair roots are present and hash-bound')

In [ ]:
# Run/resume the single source-of-truth Stage-2 trainer across the separate roots; no media is copied or merged.
trainer = EXTRACT_ROOT / 'ml/camera_coach/train_silver_actions.py'
config = EXTRACT_ROOT / 'ml/camera_coach/configs/silver_actions_stage2_colab.json'
command = [sys.executable, str(trainer), '--config', str(config), '--stage1-root', str(STAGE1_ROOT), '--stage1-sha256', EXPECTED_STAGE1_ENCODER_SHA256, '--run-dir', str(RUN_ROOT), '--device', 'auto']
for pair_root, receipt_sha256 in zip(PAIR_ROOTS, PAIR_RECEIPT_SHA256S):
    command += ['--data-root', str(pair_root), '--receipt-sha256', receipt_sha256]
if RESUME:
    command += ['--resume', 'latest']
subprocess.run(command, check=True)

In [ ]:
# The only final downloadable deliverables are this candidate artifact and its receipt.
receipt = json.loads((RUN_ROOT / 'receipt.json').read_text(encoding='utf-8'))
print({'status': receipt['status'], 'fit': receipt['fit'], 'pair_roots': receipt['source']['roots'], 'trainable_heads': receipt['trainable_heads'], 'stage1_encoder': receipt['stage1_encoder'], 'last_checkpoint': receipt['last_checkpoint']})
if receipt['status'] != 'complete':
    print('Run is resumable; set RESUME=True and rerun the trainer cell.')
else:
    final_artifact = RUN_ROOT / receipt['final_artifact']['path']
    final_receipt = RUN_ROOT / 'receipt.json'
    print('FINAL ARTIFACT:', final_artifact, receipt['final_artifact']['sha256'])
    print('FINAL RECEIPT:', final_receipt)
    from google.colab import files
    files.download(str(final_artifact))
    files.download(str(final_receipt))

## Boundary

`candidate-final.pt` is a disposable CandidateA state with only silver-action heads fitted. It is not human-gold evidence, a calibrated or release model, a Core ML export, or a redistribution artifact. Keep every pair root and its receipt hash, pair media, Stage-1 run, checkpoints, and receipt on Drive; do not use this run as validation/test evidence.